In [0]:
dbutils.widgets.removeAll()

In [0]:
from pathlib import Path
import json

# Absolute workspace configuration
ROOT_DIR = Path("/Workspace/Repos/logi@openhealthagents.org/alphaesai/ClaimsProcessing")

In [0]:
def build_gold_payload(file_id: str, client_id: str, layout_id: str = "834") -> str:
    """Generates Gold layer payload for GenericSubGroupProcessing."""
    
    gold_entry = {
        "FileId": file_id,
        "ClientID": client_id,
        "SourceContainer": "claimsprocessing.silver.member",
        "FileLayoutID": layout_id,
        "FileLayoutDescription": f"Standard{layout_id}",
        "TargetContainer": "claimsprocessing.gold.qualitymeasure",
        "ConfigFilePath": f"{ROOT_DIR}/DimQualityMeasure/Gold/Config",
        "ConfigFileName": "dimQualityMeasure.json",
        "DataUpdateFilePath": f"{ROOT_DIR}/DimQualityMeasure/Gold/DataUpdate",
        "DataUpdateFileName": "dimQualityMeasureupdate.sql",
        "DataProcessingFilePath": f"{ROOT_DIR}/DimQualityMeasure/Gold/DataProcessing",
        "DataProcessingFileName": "dimQualityMeasureprocessing.sql"
    }
    
    return json.dumps({"FileIds": [gold_entry]})

In [0]:
def trigger_gold_notebook(gold_payload: str):
    """Triggers Gold layer notebook for dimension processing."""
    gold_notebooks_base = f"{ROOT_DIR}/DimQualityMeasure/Gold/Notebooks"
    
    try:
        print("\n=== Triggering GenericSubGroupProcessing (Gold Layer) ===")
        dbutils.notebook.run(
            f"{gold_notebooks_base}/GenericSubGroupProcessing", 
            600, 
            {"GoldProcessingJSON": gold_payload}
        )
        print("GenericSubGroupProcessing completed successfully")
        
    except Exception as e:
        print(f"Gold layer processing failed: {e}")
        raise

In [0]:
def main():
    """Main orchestration - triggers Gold layer processing only."""
    
    # Example parameters - modify as needed
    file_id = "000000001"
    client_id = "HOSPITAL01"
    layout_id = "834"
    
    print(f"\n=======================================================")
    print(f"Starting QualityMeasure Gold Layer Processing")
    print(f"FileID: {file_id}, ClientID: {client_id}")
    print(f"=======================================================")
    
    try:
        # Build Gold payload and trigger Gold layer
        gold_payload = build_gold_payload(file_id, client_id, layout_id)
        trigger_gold_notebook(gold_payload)
        
        print(f"\n✓ QualityMeasure Gold processing completed successfully\n")
        
    except Exception as e:
        print(f"✗ Pipeline failed: {e}")
        raise

In [0]:
if __name__ == "__main__":
    main()